# Modulo 9 · Lezione 1 — Acquisizione dei Dati

**Progetto Finale: Sistema Intelligente di Predizione ed Esposizione di Modelli**

---

## Contesto del progetto

Lungo questo modulo costruiremo un sistema completo end-to-end:

```
Dati → Pulizia → SQL → ML → DL → API REST → Documentazione → Deployment
```

**Dataset conduttore:** Wine Quality (UCI Machine Learning Repository)
- 1.599 vini rossi con 11 caratteristiche fisico-chimiche
- Obiettivo: predire la qualità (score 0-10) e classificare se è un "buon vino" (≥7)
- URL pubblico scaricabile → ideale per dimostrare la fase di acquisizione reale

---

## Cosa vedremo oggi?

| # | Argomento |
|---|------|
| 1 | Fonti di dati: tipi e strategie |
| 2 | Download con `requests` |
| 3 | Leggere CSV, JSON e API REST pubbliche |
| 4 | Validazione iniziale del dataset |
| 5 | Salvare i dati grezzi in modo riproducibile

In [ ]:
import requests
import pandas as pd
import numpy as np
import json
import os
import hashlib
import matplotlib.pyplot as plt
from pathlib import Path
from datetime import datetime

# Creare la struttura di directory del progetto
for directorio in ['data/raw', 'data/processed', 'data/db',
                   'models', 'api', 'logs', 'docs']:
    Path(directorio).mkdir(parents=True, exist_ok=True)

print("Struttura del progetto creata:")
for d in Path('.').iterdir():
    if d.is_dir() and not d.name.startswith('.'):
        print(f"  {d}/")

---
## 1. Fonti di dati: tipi e strategie

```
Tipi di fonte:                      Strumento principale:
  File CSV/Excel/JSON               → pandas.read_*
  URL pubblico (CSV, JSON)          → requests + pandas
  API REST                          → requests
  Database SQL                      → sqlite3, psycopg2, SQLAlchemy
  Web scraping                      → BeautifulSoup, Scrapy, Playwright
  Streaming / Kafka                 → kafka-python
  Cloud (S3, GCS, Azure Blob)       → boto3, google-cloud, azure-storage

Principi dell'acquisizione professionale:
  1. Riproducibile → stesso seed, stesso URL, stesso risultato
  2. Tracciabile   → registrare fonte, data, hash del file
  3. Grezza        → salvare i dati SENZA modificarli (raw)
  4. Verificata    → controllare l'integrità (hash MD5/SHA256)
  5. Documentata   → metadati accanto al dato
```

---
## 2. Download con `requests`

In [ ]:
# URL pubblico del dataset dei vini (UCI ML Repository)
URL_VINO_TINTO  = 'https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv'
URL_VINO_BLANCO = 'https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-white.csv'

def descargar_dataset(url, ruta_local, forzar=False):
    """
    Scarica un file da un URL se non esiste già localmente.
    Restituisce i metadati del download.
    """
    ruta = Path(ruta_local)

    if ruta.exists() and not forzar:
        print(f"  Esiste già: {ruta} → uso la cache")
    else:
        print(f"  Scaricando: {url}")
        respuesta = requests.get(url, timeout=30)
        respuesta.raise_for_status()  # lancia un'eccezione se fallisce (4xx, 5xx)
        ruta.write_bytes(respuesta.content)
        print(f"  Salvato in: {ruta}")

    # Calcolare l'hash per la verifica dell'integrità
    contenido = ruta.read_bytes()
    sha256    = hashlib.sha256(contenido).hexdigest()
    tamaño    = ruta.stat().st_size

    return {
        'url':       url,
        'ruta':      str(ruta),
        'tamaño_kb': round(tamaño / 1024, 2),
        'sha256':    sha256[:16] + '...', # primi 16 caratteri
        'fecha':     datetime.now().isoformat()
    }

print("Download dei dataset di vino:")
meta_tinto  = descargar_dataset(URL_VINO_TINTO,  'data/raw/vino_tinto_raw.csv')
meta_blanco = descargar_dataset(URL_VINO_BLANCO, 'data/raw/vino_blanco_raw.csv')

print(f"\nMetadati vino rosso:  {meta_tinto}")
print(f"Metadati vino bianco: {meta_blanco}")

# Salvare i metadati del download
metadatos_descarga = {'tinto': meta_tinto, 'blanco': meta_blanco}
with open('data/raw/descarga_metadatos.json', 'w') as f:
    json.dump(metadatos_descarga, f, indent=2)
print("\nMetadati salvati in data/raw/descarga_metadatos.json")

In [ ]:
# Leggere il CSV scaricato
# Il separatore è ';' (non ',') — importante ispezionarlo prima di darlo per scontato
print("Prime righe del file grezzo:")
with open('data/raw/vino_tinto_raw.csv', 'r') as f:
    for i, linea in enumerate(f):
        print(f"  {linea.rstrip()}")
        if i >= 3:
            break

print("\n→ Separatore: ';'  (non la virgola predefinita)")

# Caricare correttamente
df_tinto  = pd.read_csv('data/raw/vino_tinto_raw.csv',  sep=';')
df_blanco = pd.read_csv('data/raw/vino_blanco_raw.csv', sep=';')

# Aggiungere la colonna tipo
df_tinto['tipo']  = 'tinto'
df_blanco['tipo'] = 'blanco'

# Combinare
df = pd.concat([df_tinto, df_blanco], ignore_index=True)

print(f"\nDataFrame combinato: {df.shape}")
print(f"  Rossi:  {len(df_tinto):,}")
print(f"  Bianchi: {len(df_blanco):,}")
print(f"  Totale:   {len(df):,}")
df.head(3)

---
## 3. Leggere JSON e API REST pubbliche

In [ ]:
# Demo: consumare una API REST pubblica
# Usiamo la API di Open Notify (posizione ISS) — gratuita e senza autenticazione

def obtener_dato_api(url, timeout=10):
    """Fa una GET a una API REST e restituisce il JSON come dict."""
    respuesta = requests.get(url, timeout=timeout)
    respuesta.raise_for_status()
    return respuesta.json()

# Astronauti nello spazio in questo momento
try:
    datos_iss = obtener_dato_api('http://api.open-notify.org/astros.json')
    print("API Open Notify — Astronauti nello spazio:")
    print(json.dumps(datos_iss, indent=2))
    df_astros = pd.DataFrame(datos_iss['people'])
    print(f"\nDataFrame: {df_astros.shape}")
    print(df_astros)
except Exception as e:
    print(f"API non disponibile ({e}) — demo con dati simulati:")
    datos_iss = {
        'people': [
            {'name': 'Oleg Kononenko', 'craft': 'ISS'},
            {'name': 'Nikolai Chub',   'craft': 'ISS'},
        ],
        'number': 2
    }
    print(json.dumps(datos_iss, indent=2))

In [ ]:
# Pattern comuni di lettura dei dati
print("Pattern di lettura con pandas:")
print("""
  # CSV con virgola
  df = pd.read_csv('archivo.csv')

  # CSV con punto e virgola
  df = pd.read_csv('archivo.csv', sep=';')

  # CSV direttamente da URL
  df = pd.read_csv('https://example.com/datos.csv')

  # Excel
  df = pd.read_excel('archivo.xlsx', sheet_name='Hoja1')

  # JSON (lista di oggetti)
  df = pd.read_json('archivo.json')

  # JSON da API (dict → DataFrame manuale)
  datos = requests.get(url).json()
  df = pd.DataFrame(datos['items'])

  # Parquet (binario compresso, molto efficiente)
  df = pd.read_parquet('archivo.parquet')

  # SQL
  import sqlite3
  conn = sqlite3.connect('base.db')
  df = pd.read_sql('SELECT * FROM tabla', conn)
""")

---
## 4. Validazione iniziale del dataset

In [ ]:
def informe_inicial(df, nombre='Dataset'):
    """Genera un report rapido di un DataFrame appena caricato."""
    print(f"\n{'='*55}")
    print(f"  REPORT INIZIALE — {nombre}")
    print(f"{'='*55}")
    print(f"  Righe:    {df.shape[0]:,}")
    print(f"  Colonne: {df.shape[1]}")
    print(f"  Memoria:  {df.memory_usage(deep=True).sum() / 1024:.1f} KB")

    print(f"\n  Tipi di dato:")
    for col, dtype in df.dtypes.items():
        nulls = df[col].isna().sum()
        print(f"    {col:30} {str(dtype):10} nulls={nulls}")

    print(f"\n  Duplicati esatti: {df.duplicated().sum()}")

    num_cols = df.select_dtypes(include='number').columns.tolist()
    if num_cols:
        print(f"\n  Statistiche numeriche (prime 5 colonne):")
        print(df[num_cols[:5]].describe().round(3).to_string())

informe_inicial(df, 'Vini combinati (rosso + bianco)')

In [ ]:
# Ispezione visiva rapida
fig, axes = plt.subplots(3, 4, figsize=(15, 10))
cols_num = df.select_dtypes(include='number').columns.tolist()

for ax, col in zip(axes.ravel(), cols_num):
    df_t = df[df['tipo']=='tinto'][col]
    df_b = df[df['tipo']=='blanco'][col]
    ax.hist(df_t, bins=30, alpha=0.6, color='firebrick',  label='Rosso',  density=True)
    ax.hist(df_b, bins=30, alpha=0.6, color='steelblue', label='Bianco', density=True)
    ax.set_title(col, fontsize=9)
    ax.set_xlabel(''); ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3)

# Nascondere il subplot in più
axes.ravel()[-1].set_visible(False)

plt.suptitle('Distribuzioni per variabile e tipo di vino', fontsize=14)
plt.tight_layout()
plt.show()

# Distribuzione della variabile obiettivo
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df['quality'].value_counts().sort_index().plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('Distribuzione della qualità (totale)'); axes[0].set_xlabel('Punteggio')
axes[0].grid(True, alpha=0.3, axis='y')

for tipo, color in [('tinto','firebrick'), ('blanco','steelblue')]:
    df[df['tipo']==tipo]['quality'].value_counts().sort_index().plot(
        kind='line', ax=axes[1], color=color, lw=2, marker='o', label=tipo)
axes[1].set_title('Qualità per tipo'); axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Distribuzione della qualità:")
print(df['quality'].value_counts().sort_index().to_string())

---
## 5. Salvare i dati grezzi in modo riproducibile

In [ ]:
# Salvare il dataset combinato SENZA modificarlo
# Usare Parquet: più efficiente del CSV per dati tabulari
ruta_raw = 'data/raw/vinos_combinados_raw.parquet'
df.to_parquet(ruta_raw, index=False)
print(f"Dataset grezzo salvato: {ruta_raw}")

# Anche in CSV come backup leggibile
df.to_csv('data/raw/vinos_combinados_raw.csv', index=False)
print("Backup CSV: data/raw/vinos_combinados_raw.csv")

# Salvare i metadati del dataset
meta_dataset = {
    'nombre':        'Wine Quality (UCI)',
    'fuente':        'https://archive.ics.uci.edu/ml/datasets/wine+quality',
    'citacion':      'Cortez et al., 2009. Wine Quality.',
    'fecha_descarga': datetime.now().isoformat(),
    'filas':         int(df.shape[0]),
    'columnas':      int(df.shape[1]),
    'columnas_lista': df.columns.tolist(),
    'variable_objetivo': 'quality',
    'tipos': {'tinto': int(df['tipo'].eq('tinto').sum()),
              'blanco': int(df['tipo'].eq('blanco').sum())},
    'calidad_min': int(df['quality'].min()),
    'calidad_max': int(df['quality'].max()),
    'sha256_raw': hashlib.sha256(Path(ruta_raw).read_bytes()).hexdigest()[:16]
}

with open('data/raw/dataset_metadatos.json', 'w', encoding='utf-8') as f:
    json.dump(meta_dataset, f, indent=2, ensure_ascii=False)

print(f"\nMetadati del dataset:")
print(json.dumps(meta_dataset, indent=2))

In [ ]:
# Verificare che i dati possano essere letti correttamente
df_verificado = pd.read_parquet('data/raw/vinos_combinados_raw.parquet')
assert df_verificado.shape == df.shape, "Errore: le shape non coincidono"
assert list(df_verificado.columns) == list(df.columns), "Errore: colonne diverse"
print(f"✓ Verifica corretta: {df_verificado.shape}")
print(f"  Colonne: {list(df_verificado.columns)}")

---
## Esercitazione pratica

### Esercizio 1 — Ottenere dati da un'altra API pubblica

**Consegna:** La API di Open Meteo (https://open-meteo.com) è gratuita e restituisce dati meteorologici.

```
URL: https://api.open-meteo.com/v1/forecast
Parametri: latitude=43.35, longitude=-8.41, hourly=temperature_2m&forecast_days=3
(coordinate di A Coruña)
```

1. Fai una GET all'URL con `requests.get(url, params=dict)`
2. Converti il JSON in DataFrame
3. Mostra le prime righe e traccia il grafico della temperatura per ora

In [ ]:
# ✏️ IL TUO CODICE QUI


<details>
<summary>💡 Vedi soluzione — Esercizio 1</summary>

```python
import requests
import pandas as pd
import matplotlib.pyplot as plt

url = 'https://api.open-meteo.com/v1/forecast'
params = {
    'latitude':  43.35,
    'longitude': -8.41,
    'hourly':    'temperature_2m',
    'forecast_days': 3
}

try:
    resp = requests.get(url, params=params, timeout=10)
    resp.raise_for_status()
    datos = resp.json()

    df_met = pd.DataFrame({
        'tiempo':      pd.to_datetime(datos['hourly']['time']),
        'temperatura': datos['hourly']['temperature_2m']
    })

    print(f"Dati ottenuti: {df_met.shape}")
    print(df_met.head())

    fig, ax = plt.subplots(figsize=(12, 4))
    ax.plot(df_met['tiempo'], df_met['temperatura'], 'steelblue', lw=2)
    ax.fill_between(df_met['tiempo'], df_met['temperatura'], alpha=0.2)
    ax.set_title('Temperatura prevista a A Coruña (prossime 72h)')
    ax.set_xlabel('Ora'); ax.set_ylabel('°C')
    ax.grid(True, alpha=0.3)
    plt.xticks(rotation=45)
    plt.tight_layout(); plt.show()

except Exception as e:
    print(f"Errore: {e}")
```
</details>

---
### Esercizio 2 — Gestione robusta degli errori HTTP

**Consegna:** In produzione gli errori di rete sono inevitabili. Migliora la funzione `descargar_dataset` per:

1. Riprovare fino a 3 volte in caso di fallimento (con attesa tra i tentativi)
2. Registrare ogni tentativo in un log (`logs/descargas.log`)
3. Restituire `None` se i tentativi si esauriscono (senza lanciare eccezioni)

Suggerimento: usa `time.sleep()` e un ciclo `for`.

In [ ]:
# ✏️ IL TUO CODICE QUI


<details>
<summary>💡 Vedi soluzione — Esercizio 2</summary>

```python
import requests
import time
import logging
from pathlib import Path
from datetime import datetime

logging.basicConfig(
    filename='logs/descargas.log',
    level=logging.INFO,
    format='%(asctime)s %(levelname)s %(message)s'
)

def descargar_con_reintentos(url, ruta_local, max_reintentos=3, espera=2):
    ruta = Path(ruta_local)
    for intento in range(1, max_reintentos + 1):
        try:
            logging.info(f"Tentativo {intento}/{max_reintentos}: {url}")
            print(f"Tentativo {intento}/{max_reintentos}: {url}")
            resp = requests.get(url, timeout=15)
            resp.raise_for_status()
            ruta.parent.mkdir(parents=True, exist_ok=True)
            ruta.write_bytes(resp.content)
            logging.info(f"OK: {ruta} ({len(resp.content)/1024:.1f} KB)")
            print(f"  ✓ Scaricato: {len(resp.content)/1024:.1f} KB")
            return ruta
        except requests.RequestException as e:
            logging.warning(f"Tentativo fallito {intento}: {e}")
            print(f"  ✗ Errore: {e}")
            if intento < max_reintentos:
                print(f"  Attendo {espera}s prima del prossimo tentativo...")
                time.sleep(espera)
    logging.error(f"Tutti i tentativi sono falliti: {url}")
    print(f"✗ Impossibile scaricare dopo {max_reintentos} tentativi")
    return None

# Provare
resultado = descargar_con_reintentos(
    'https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv',
    'data/raw/test_reintentos.csv'
)
print(f"\nRisultato: {resultado}")
print("Log salvato in logs/descargas.log")
```
</details>

---
### Esercizio 3 — Validatore di schema

**Consegna:** Crea una funzione `validar_esquema(df, esquema_esperado)` dove `esquema_esperado` è un dict `{colonna: dtype_atteso}`. La funzione deve:

1. Verificare che tutte le colonne attese esistano
2. Verificare che i tipi coincidano (almeno la famiglia: `float`, `int`, `object`)
3. Verificare che non ci siano valori nulli
4. Restituire un dict `{'ok': bool, 'errori': list[str]}`

In [ ]:
# ✏️ IL TUO CODICE QUI


<details>
<summary>💡 Vedi soluzione — Esercizio 3</summary>

```python
import pandas as pd
import numpy as np

def validar_esquema(df, esquema_esperado):
    errores = []

    for col, tipo_esperado in esquema_esperado.items():
        # 1. La colonna esiste
        if col not in df.columns:
            errores.append(f"Colonna mancante: '{col}'")
            continue

        # 2. Tipo di dato
        dtype_real = df[col].dtype
        if tipo_esperado == 'float' and not pd.api.types.is_float_dtype(dtype_real):
            errores.append(f"'{col}': atteso float, trovato {dtype_real}")
        elif tipo_esperado == 'int' and not pd.api.types.is_integer_dtype(dtype_real):
            errores.append(f"'{col}': atteso int, trovato {dtype_real}")
        elif tipo_esperado == 'object' and not pd.api.types.is_object_dtype(dtype_real):
            errores.append(f"'{col}': atteso object, trovato {dtype_real}")

        # 3. Nessun nullo
        n_nulos = df[col].isna().sum()
        if n_nulos > 0:
            errores.append(f"'{col}': {n_nulos} valori nulli")

    return {'ok': len(errores) == 0, 'errores': errores}

# Provare con il dataset dei vini
df = pd.read_parquet('data/raw/vinos_combinados_raw.parquet')

esquema = {
    'fixed acidity':       'float',
    'volatile acidity':    'float',
    'citric acid':         'float',
    'residual sugar':      'float',
    'chlorides':           'float',
    'free sulfur dioxide': 'float',
    'total sulfur dioxide':'float',
    'density':             'float',
    'pH':                  'float',
    'sulphates':           'float',
    'alcohol':             'float',
    'quality':             'int',
    'tipo':                'object',
}

resultado = validar_esquema(df, esquema)
print(f"Validazione: {'✓ OK' if resultado['ok'] else '✗ FALLITA'}")
if resultado['errores']:
    for e in resultado['errores']:
        print(f"  - {e}")
else:
    print("  Tutti i tipi e i nulli verificati correttamente")
```
</details>

---
### Esercizio 4 — Download incrementale

**Consegna:** Nei sistemi di produzione i dati arrivano a lotti. Simula un download incrementale:

1. Dividi il dataset in 3 "lotti" (batch_001, batch_002, batch_003)
2. Salva ogni lotto come `data/raw/batch_NNN.parquet`
3. Crea una funzione `cargar_todos_los_lotes(directorio)` che li combini
4. Verifica che il risultato sia uguale al dataset originale

In [ ]:
# ✏️ IL TUO CODICE QUI


<details>
<summary>💡 Vedi soluzione — Esercizio 4</summary>

```python
import pandas as pd
import numpy as np
from pathlib import Path

df = pd.read_parquet('data/raw/vinos_combinados_raw.parquet')

# Dividere in 3 lotti
n = len(df)
lotes = np.array_split(df, 3)

for i, lote in enumerate(lotes, 1):
    ruta = f'data/raw/batch_{i:03d}.parquet'
    lote.to_parquet(ruta, index=False)
    print(f"Lotto {i:03d}: {len(lote):,} righe → {ruta}")

def cargar_todos_los_lotes(directorio):
    rutas = sorted(Path(directorio).glob('batch_*.parquet'))
    if not rutas:
        raise FileNotFoundError(f"Nessun lotto in {directorio}")
    lotes = [pd.read_parquet(r) for r in rutas]
    print(f"Caricati {len(lotes)} lotti: {[len(l) for l in lotes]}")
    return pd.concat(lotes, ignore_index=True)

df_reconstruido = cargar_todos_los_lotes('data/raw')
print(f"\nDataset ricostruito: {df_reconstruido.shape}")

assert df_reconstruido.shape == df.shape, "Shape diverse"
assert (df_reconstruido.values == df.values).all(), "Contenuto diverso"
print("✓ Verifica OK: dataset ricostruito identico all'originale")
```
</details>

---
### Esercizio 5 (sfida) — Pipeline di ingestione completa

**Consegna:** Crea una classe `IngestaPipeline` che incapsuli l'intero processo:

```python
pipeline = IngestaPipeline(
    fuentes={'tinto': URL_TINTO, 'blanco': URL_BLANCO},
    directorio_raw='data/raw',
    esquema_esperado={'quality': 'int', 'alcohol': 'float'}
)
df = pipeline.ejecutar()  # scarica + combina + valida + salva + restituisce
```

In [ ]:
# ✏️ IL TUO CODICE QUI


<details>
<summary>💡 Vedi soluzione — Esercizio 5</summary>

```python
import requests, json, hashlib, logging
import pandas as pd
from pathlib import Path
from datetime import datetime

class IngestaPipeline:
    def __init__(self, fuentes, directorio_raw, esquema_esperado=None):
        self.fuentes    = fuentes  # dict {nome: url}
        self.dir_raw    = Path(directorio_raw)
        self.esquema    = esquema_esperado or {}
        self.dir_raw.mkdir(parents=True, exist_ok=True)
        self.log        = []

    def _descargar(self, nombre, url):
        ruta = self.dir_raw / f'{nombre}_raw.csv'
        if not ruta.exists():
            r = requests.get(url, timeout=30)
            r.raise_for_status()
            ruta.write_bytes(r.content)
        self.log.append({'accion': 'descarga', 'nombre': nombre, 'ruta': str(ruta)})
        return ruta

    def _cargar(self, nombre, ruta):
        df = pd.read_csv(ruta, sep=';')
        df['tipo'] = nombre
        return df

    def _validar(self, df):
        errores = []
        for col, tipo in self.esquema.items():
            if col not in df.columns:
                errores.append(f"Colonna mancante: {col}")
            elif df[col].isna().any():
                errores.append(f"Nulli in: {col}")
        return errores

    def ejecutar(self):
        dataframes = []
        for nombre, url in self.fuentes.items():
            ruta = self._descargar(nombre, url)
            df   = self._cargar(nombre, ruta)
            dataframes.append(df)
            print(f"  {nombre}: {df.shape}")

        df_final = pd.concat(dataframes, ignore_index=True)

        errores = self._validar(df_final)
        if errores:
            raise ValueError(f"Validazione fallita: {errores}")

        ruta_out = self.dir_raw / 'ingesta_final.parquet'
        df_final.to_parquet(ruta_out, index=False)

        meta = {'filas': len(df_final), 'fecha': datetime.now().isoformat(), 'log': self.log}
        (self.dir_raw / 'ingesta_meta.json').write_text(json.dumps(meta, indent=2))

        print(f"\nIngestione completata: {df_final.shape} → {ruta_out}")
        return df_final


pipeline = IngestaPipeline(
    fuentes={
        'tinto':  'https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv',
        'blanco': 'https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-white.csv',
    },
    directorio_raw='data/raw',
    esquema_esperado={'quality': 'int', 'alcohol': 'float'}
)
df = pipeline.ejecutar()
print(df.head())
```
</details>

---
## Riepilogo

```python
import requests
import pandas as pd
from pathlib import Path

# Scaricare un file
resp = requests.get(url, timeout=30)
resp.raise_for_status()
Path('data/raw/archivo.csv').write_bytes(resp.content)

# Leggere CSV con separatore non standard
df = pd.read_csv('archivo.csv', sep=';')

# Consumare una API REST
datos = requests.get(url, params={'key': 'value'}).json()
df = pd.DataFrame(datos['items'])

# Salvare i dati grezzi (Parquet = efficiente)
df.to_parquet('data/raw/dataset_raw.parquet', index=False)

# Verificare l'integrità
import hashlib
sha = hashlib.sha256(Path('archivo.parquet').read_bytes()).hexdigest()

# Principi:
# 1. Salvare i dati GREZZI senza modificarli
# 2. Registrare fonte + data + hash
# 3. Gestire gli errori di rete (tentativi ripetuti)
# 4. Validare lo schema al caricamento
```

## Cosa vedremo nella prossima lezione?
Nella **Lezione 2** puliremo e prepareremo i dati: trattamento degli outlier, encoding, feature engineering e salvataggio del dataset pronto per la modellazione.